# ENIAC A/B Test Project

## 1. Experimental Design

### 1. Would you include all variants in the experiment (A, B, C and D)?

Yes, all four variants (A, B, C, and D) should be included in the experiment because the goal is to compare all possible button designs and identify the best-performing version.

The four versions differ in both color and text:
- White "SHOP NOW"
- Red "SHOP NOW"
- White "SEE DEALS"
- Red "SEE DEALS"

Testing all variants allows us to evaluate both factors simultaneously and understand which combination performs best.

---

### 2. What is the business value of this experiment?

The experiment can increase the click-through rate (CTR), which may lead to higher conversions and revenue. Improving user engagement with the "Shop Now" button directly supports business growth.

---

### 3. Which main metric would you choose?

The main metric is the click-through rate (CTR), defined as the number of clicks divided by the number of visits.

---

### 4. Which additional metrics would you track?

Additional metrics include conversion rate, bounce rate, and time spent on the page. These help ensure that improving CTR does not negatively impact user experience.

---

### 5. How would you define the null and alternative hypotheses?

H0 (null hypothesis): The click-through rate is the same for all variants.

H1 (alternative hypothesis): At least one variant has a different click-through rate.

---

### 6. What significance level would you set?

I would set the significance level at 0.05, which is a standard threshold in statistical testing.

---

### 7. What is the minimum detectable effect (MDE)?

The minimum detectable effect (MDE) was set to 20%. This means that the experiment was designed to detect relatively large differences in CTR between variants.

---

### 8. Would you use a custom platform or external tools?

The experiment can be conducted using external tools such as A/B Tasty, without requiring a custom-built platform.

---



## 2. Next Steps

- Explore the data from the experiment  
- Perform a Chi-Square test  

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns

# -------------------------
# Load files from local project folder
# -------------------------

import pandas as pd

base_url = "https://raw.githubusercontent.com/zahraghaedianroonizi/ENIAC_AB_Test/main/"

df_A = pd.read_csv(base_url + "eniac_a.csv")
df_B = pd.read_csv(base_url + "eniac_b.csv")
df_C = pd.read_csv(base_url + "eniac_c.csv")
df_D = pd.read_csv(base_url + "eniac_d.csv")





## Extract Click Data

The dataset contains multiple tracked elements, so we need to filter only the main CTA buttons used in the experiment.

- Version A → White "SHOP NOW"
- Version B → Red "SHOP NOW"
- Version C → White "SEE DEALS"
- Version D → Red "SEE DEALS"

The number of clicks for each main button is extracted and will be used for CTR and Chi-Square analysis.

In [ ]:
clicks_A = df_A.loc[df_A["Name"] == "SHOP NOW", "No. clicks"].iloc[0]
clicks_B = df_B.loc[df_B["Name"] == "SHOP NOW", "No. clicks"].iloc[0]
clicks_C = df_C.loc[df_C["Name"] == "SEE DEALS", "No. clicks"].iloc[0]
clicks_D = df_D.loc[df_D["Name"] == "SEE DEALS", "No. clicks"].iloc[0]

clicks_A, clicks_B, clicks_C, clicks_D

In [ ]:
df_A.iloc[1, -1]

In [ ]:
df_B.iloc[1, -1]

In [ ]:
df_C.iloc[1, -1]

In [ ]:
df_D.iloc[1, -1]

In [ ]:
visits_A = 25326
visits_B = 24747
visits_C = 24876
visits_D = 25233

In [ ]:
no_click_A = visits_A - clicks_A
no_click_B = visits_B - clicks_B
no_click_C = visits_C - clicks_C
no_click_D = visits_D - clicks_D

In [ ]:
clicks = [clicks_A, clicks_B, clicks_C, clicks_D]
no_clicks = [no_click_A, no_click_B, no_click_C, no_click_D]

observed_results = pd.DataFrame(
    data=[clicks, no_clicks],
    columns=["Version_A", "Version_B", "Version_C", "Version_D"],
    index=["Click", "No_Click"]
)

observed_results

In [ ]:
chisq, pvalue, df, expected = stats.chi2_contingency(observed_results)

In [ ]:
df

In [ ]:
expected

In [ ]:
pvalue

In [ ]:
alpha = 0.05

if pvalue > alpha:
    print("Do not reject the null hypothesis")
else:
    print("Reject the null hypothesis")

### Interpretation of the Chi-Square Test Result

The p-value obtained from the Chi-Square test is extremely small (`2.716e-48`), which is far below the significance level of `α = 0.05`.

Since `p-value < alpha`, we reject the null hypothesis (H₀).

This means that there is a statistically significant difference between the four website versions (A, B, C, and D). Therefore, not all versions perform equally in terms of user click behavior.

The differences observed in clicks are unlikely to be due to random chance alone.

In [ ]:
possible_combinations = 6
alpha_post_hoc = alpha / possible_combinations
alpha_post_hoc

In [ ]:
# click-through rates

ctr_A = clicks_A / visits_A
ctr_B = clicks_B / visits_B
ctr_C = clicks_C / visits_C
ctr_D = clicks_D / visits_D

# display as DataFrame

rates = [ctr_A, ctr_B, ctr_C, ctr_D]
names = ["Version_A", "Version_B", "Version_C", "Version_D"]

ctr_df = pd.DataFrame({
    "rates": rates,
    "names": names
})

ctr_df.sort_values("rates", ascending=False)

### Click Through Rate (CTR) Comparison

To determine the best-performing version, we calculated the Click Through Rate (CTR) for each website version.

CTR is calculated using the following formula:

CTR = Number of Clicks / Number of Visits

The results were:

- Version C → 0.021185
- Version A → 0.020216
- Version B → 0.011355
- Version D → 0.007649

Version C has the highest CTR, which means it generated the highest proportion of clicks relative to visits.

At this stage, Version C appears to be the strongest candidate for the winning version.

In [ ]:
observed_results.columns

In [ ]:
observed_results.loc[:, ["Version_B", "Version_A"]]

In [ ]:
# empty dictionary to fill with our results

stat_significant_dict = {
    "Version_A": [],
    "Version_B": [],
    "Version_C": [],
    "Version_D": []
}

# compare each version to each other version

for i in observed_results.columns:
    for j in observed_results.columns:

        # use scipy to find the p-value of each pair
        chisq, pvalue, df, expected = stats.chi2_contingency(
            observed_results.loc[:, [i, j]],
            correction=False
        )

        # boolean:
        # if p-value is lower than alpha,
        # our result is statistically significant

        stat_significant_dict[i].append(
            pvalue < alpha_post_hoc
        )

In [ ]:
# Create a heatmap from the pairwise significance results

red_green_palette = sns.diverging_palette(10, 120, n=2, s=70, l=50)

ax = sns.heatmap(stat_significant_df, cmap=red_green_palette)

# Manually specify colorbar labels
colorbar = ax.collections[0].colorbar
colorbar.set_ticks([0.25, 0.75])
colorbar.set_ticklabels(["False", "True"])

# Add title
ax.set_title("Statistical Significance in Pairwise Tests: True or False?", pad=10);

In [ ]:
stat_significant_df = pd.DataFrame(
    stat_significant_dict,
    index=observed_results.columns
)

stat_significant_df

## Final Result

- Chi-square test showed a statistically significant difference between the website versions because the p-value was extremely small (much smaller than α = 0.05).

- Therefore, we reject the null hypothesis (H₀) and conclude that button design affects user click behavior.

- Based on the CTR ranking:

1. Version C → highest performance (best CTR)
2. Version A
3. Version B
4. Version D → weakest performance

- Post hoc pairwise tests confirmed that several version pairs are significantly different, especially showing that the white buttons performed much better than the darker versions.

## Business Recommendation

Version C ("SEE DEALS" with white button) should be selected as the winning version because it achieved the highest click-through rate and showed strong statistical significance.

This version is the best candidate for implementation on the website to maximize customer engagement and conversions.